# Cricket Ball Detection - YOLOv8 Training (Google Colab)

Steps:
1. Mount Google Drive
2. Install Ultralytics
3. Point to your dataset (Roboflow export: train/valid/test with images + labels)
4. Train YOLOv8n for 50 epochs with early stopping
5. Auto-clear GPU VRAM after training

**Before running:** Update `DATASET_PATH` in the config cell to your dataset folder in Google Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

## 3. Config — update these paths

In [ ]:
import os

# ==== EDIT THIS ====
# Path to the dataset ZIP file in Google Drive
DATASET_ZIP = "/content/drive/MyDrive/cricket_ball_dataset.zip"
# ====================

# Dataset is extracted here on Colab's local disk (much faster than reading from Drive directly)
DATASET_PATH = "/content/dataset"
DATA_YAML = os.path.join(DATASET_PATH, "data.yaml")

MODEL = "yolov8n.pt"      # nano model, fast + good for Colab free GPU
EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 15              # early stopping patience
PROJECT_DIR = "/content/runs/train"
RUN_NAME = "cricket_ball_yolov8n"

print("Zip file exists:", os.path.exists(DATASET_ZIP))

## 3a. Unzip dataset to local Colab disk

Extracting to `/content/dataset` (local SSD) instead of training directly from Drive is much faster and avoids I/O bottlenecks/timeouts during training.

In [ ]:
import zipfile
import shutil

if os.path.exists(DATASET_PATH):
    shutil.rmtree(DATASET_PATH)
os.makedirs(DATASET_PATH, exist_ok=True)

with zipfile.ZipFile(DATASET_ZIP, "r") as zip_ref:
    zip_ref.extractall(DATASET_PATH)

print("Extracted contents:")
print(os.listdir(DATASET_PATH))

# Handle case where zip contains a single nested subfolder (common Roboflow export quirk)
extracted_items = os.listdir(DATASET_PATH)
if len(extracted_items) == 1 and os.path.isdir(os.path.join(DATASET_PATH, extracted_items[0])):
    nested_folder = os.path.join(DATASET_PATH, extracted_items[0])
    for item in os.listdir(nested_folder):
        shutil.move(os.path.join(nested_folder, item), os.path.join(DATASET_PATH, item))
    shutil.rmtree(nested_folder)
    print("Flattened nested folder. New contents:")
    print(os.listdir(DATASET_PATH))

print("\nUsing data.yaml at:", DATA_YAML)
print("Exists:", os.path.exists(DATA_YAML))

## 3b. Fix Roboflow data.yaml paths (important for Colab)

Roboflow exports often have relative paths (`../train/images`) that break outside the original folder structure. This cell rewrites `data.yaml` with absolute Colab paths so training doesn't fail with a "dataset not found" error.

In [ ]:
import yaml

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

data_cfg["train"] = os.path.join(DATASET_PATH, "train", "images")
data_cfg["val"] = os.path.join(DATASET_PATH, "valid", "images")
if os.path.exists(os.path.join(DATASET_PATH, "test", "images")):
    data_cfg["test"] = os.path.join(DATASET_PATH, "test", "images")

fixed_yaml_path = "/content/data_fixed.yaml"
with open(fixed_yaml_path, "w") as f:
    yaml.dump(data_cfg, f)

DATA_YAML = fixed_yaml_path
print("Fixed data.yaml written to:", DATA_YAML)
print(data_cfg)

## 4. GPU check + memory reset helper

In [ ]:
import gc
import torch

def free_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
    print("GPU VRAM memory cleared.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    DEVICE = 0
else:
    print("No GPU found — go to Runtime > Change runtime type > GPU")
    DEVICE = "cpu"

free_gpu_memory()

## 5. Train YOLOv8 (50 epochs, early stopping enabled)

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=2,
    patience=PATIENCE,       # early stopping
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    amp=True,
    cache=True,               # Colab has more RAM, ok to cache
    plots=True,
    save=True,
    save_period=5,
)

print("Training complete.")

## 6. Validate on test set

In [ ]:
try:
    metrics = model.val(data=DATA_YAML, split="test", device=DEVICE)
    print(metrics)
except Exception as e:
    print("Test split validation skipped/failed:", e)

## 7. Save best model to Google Drive

In [ ]:
import shutil

# Saved next to the zip file in Drive
drive_folder = os.path.dirname(DATASET_ZIP)
best_model_path = os.path.join(PROJECT_DIR, RUN_NAME, "weights", "best.pt")
drive_save_path = os.path.join(drive_folder, "cricket_ball_best.pt")

if os.path.exists(best_model_path):
    shutil.copy(best_model_path, drive_save_path)
    print(f"Best model saved to Google Drive: {drive_save_path}")
else:
    print("best.pt not found — check training logs above.")

## 8. Free GPU VRAM

In [ ]:
del model
free_gpu_memory()